# NB09b — 배송 운영 구역 분류 (Delivery Zone Classification)

NB09의 통합 제약 점수를 바탕으로 각 H3 셀을 **배송 운영 구역**으로 분류합니다.
이 결과는 NB10 후보지 점수 산정의 입력으로 사용됩니다.

## 구역 정의

| 구역 | 조건 | 설명 |
|------|------|------|
| 🟢 **하이브리드** | drone ≥ 0.65 & robot ≥ 0.45 | 드론·로봇 동시 운용 가능 |
| 🔵 **드론우선** | drone ≥ 0.65 & robot < 0.45 | 드론 배송 최적, 로봇 보조 |
| 🟠 **로봇우선** | robot ≥ 0.45 & drone < 0.55 | 로봇 배송 최적, 드론 제약 |
| 🟡 **보완검토** | drone ≥ 0.50 or robot ≥ 0.30 | 조건부 운용 가능 |
| ⛔ **부적합** | 공식 제외 또는 운용성 부족 | 배송 거점 후보 제외 |

## 종합 점수

```
drone_score            = 0.30 × airspace + 0.25 × obstacle + 0.20 × terrain
                       + 0.15 × weather  + 0.10 × noise_proxy

robot_score            = score_robot (Ra)

service_priority_score = 0.45 × Ds + 0.35 × drone_score + 0.20 × robot_score
```

In [1]:
import warnings
import json
import numpy as np
import pandas as pd
import geopandas as gpd
from pathlib import Path
from datetime import datetime
from shapely.geometry import mapping

warnings.filterwarnings('ignore')

_cwd = Path.cwd()
if _cwd.name == '02_analysis':
    BASE = _cwd.parent
elif (_cwd / '02_analysis').exists():
    BASE = _cwd
else:
    BASE = _cwd.parent
PROC = BASE / 'processed'

print(f'Project root : {BASE}')
print(f'Processed    : {PROC}')
print(f'Run time     : {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')


Project root : C:\Users\jimin\Desktop\1_BITAmin_16기\1_Seongnam_reset
Processed    : C:\Users\jimin\Desktop\1_BITAmin_16기\1_Seongnam_reset\processed
Run time     : 2026-05-12 09:40:21


In [2]:
grid_path = PROC / 'constraint_grid.gpkg'
if not grid_path.exists():
    raise FileNotFoundError(
        'Run NB09_constraint_layer_scoring.ipynb first.\n'
        f'Expected: {grid_path}'
    )

gdf = gpd.read_file(grid_path)
print(f'Constraint grid: {len(gdf)} cells   CRS={gdf.crs}')

# Alias h3_id → h3_index
if 'h3_index' not in gdf.columns and 'h3_id' in gdf.columns:
    gdf['h3_index'] = gdf['h3_id']

# Column inventory
REQUIRED_COLS = [
    'h3_index', 'lat', 'lon', 'CSV_ADMI_CD', 'ADM_NM', 'GU_NM',
    'Ds', 'demand_grade',
    'score_airspace', 'score_obstacle', 'score_noise_proxy',
    'score_terrain', 'score_robot', 'score_weather',
    'constraint_score', 'constraint_grade',
]
OPTIONAL_COLS = [
    'score_airspace_proxy', 'constraint_score_with_airspace_proxy',
    'Ra', 'Ra_grade', 'airspace_data_status', 'airspace_proxy_flag',
    'hard_exclusion_flag', 'n_crossings',
]

missing_req = [c for c in REQUIRED_COLS if c not in gdf.columns]
if missing_req:
    print(f'  WARNING — missing required columns: {missing_req}')
else:
    print(f'  All required columns present.')

present_opt = [c for c in OPTIONAL_COLS if c in gdf.columns]
print(f'  Optional columns found: {present_opt}')

# Cast bool-like string columns back to bool/numeric
for col in ['hard_exclusion_flag', 'airspace_proxy_flag']:
    if col in gdf.columns:
        gdf[col] = gdf[col].map(
            {'True': True, 'False': False, True: True, False: False}
        ).fillna(False).astype(bool)

for col in gdf.columns:
    if col.startswith('score_') or col in ['Ds', 'Ra', 'constraint_score']:
        gdf[col] = pd.to_numeric(gdf[col], errors='coerce')

# Numeric coercion for demand_grade (keep as string)
print(f'\n  demand_grade unique: {sorted(gdf["demand_grade"].dropna().unique())}')
print(f'  constraint_grade unique: {sorted(gdf["constraint_grade"].dropna().unique())}')
print(f'  hard_exclusion_flag: {gdf["hard_exclusion_flag"].sum()} cells'
      if 'hard_exclusion_flag' in gdf.columns else '  hard_exclusion_flag: column absent')


Constraint grid: 1947 cells   CRS=EPSG:4326
  All required columns present.
  Optional columns found: ['Ra', 'Ra_grade', 'airspace_data_status', 'airspace_proxy_flag', 'hard_exclusion_flag', 'n_crossings']

  demand_grade unique: ['A', 'B', 'C', 'D']
  constraint_grade unique: ['A', 'B', 'C', 'D']
  hard_exclusion_flag: 1688 cells


In [3]:
# ══════════════════════════════════════════════════════════
#  drone_score — 드론 운용 적합도
# ══════════════════════════════════════════════════════════

W_AIR  = 0.30
W_OBS  = 0.25
W_TER  = 0.20
W_WEA  = 0.15
W_NOI  = 0.10
assert abs(W_AIR + W_OBS + W_TER + W_WEA + W_NOI - 1.0) < 1e-9

gdf['drone_score'] = (
    W_AIR * gdf['score_airspace'].fillna(1.0)
  + W_OBS * gdf['score_obstacle'].fillna(0.0)
  + W_TER * gdf['score_terrain'].fillna(0.6)
  + W_WEA * gdf['score_weather'].fillna(0.9)
  + W_NOI * gdf['score_noise_proxy'].fillna(0.5)
).clip(0.0, 1.0)

# Proxy variant if airspace proxy column exists
HAS_PROXY = 'score_airspace_proxy' in gdf.columns
if HAS_PROXY:
    gdf['drone_score_with_airspace_proxy'] = (
        W_AIR * gdf['score_airspace_proxy'].fillna(1.0)
      + W_OBS * gdf['score_obstacle'].fillna(0.0)
      + W_TER * gdf['score_terrain'].fillna(0.6)
      + W_WEA * gdf['score_weather'].fillna(0.9)
      + W_NOI * gdf['score_noise_proxy'].fillna(0.5)
    ).clip(0.0, 1.0)
    print('[Airspace proxy] drone_score_with_airspace_proxy 생성됨')

ds = gdf['drone_score']
print(f'\ndrone_score : mean={ds.mean():.3f}  '
      f'min={ds.min():.3f}  max={ds.max():.3f}  '
      f'std={ds.std():.3f}')
print(f'  >= 0.65: {(ds >= 0.65).sum()} cells  ({(ds >= 0.65).mean()*100:.1f}%)')
print(f'  >= 0.50: {(ds >= 0.50).sum()} cells  ({(ds >= 0.50).mean()*100:.1f}%)')



drone_score : mean=0.622  min=0.388  max=0.982  std=0.122
  >= 0.65: 489 cells  (25.1%)
  >= 0.50: 1743 cells  (89.5%)


In [4]:
# ══════════════════════════════════════════════════════════
#  robot_score — 로봇 배송 적합도
# ══════════════════════════════════════════════════════════

if 'score_robot' in gdf.columns and gdf['score_robot'].notna().sum() > 0:
    gdf['robot_score'] = gdf['score_robot'].clip(0.0, 1.0)
    robot_src = 'score_robot'
elif 'Ra' in gdf.columns and gdf['Ra'].notna().sum() > 0:
    gdf['robot_score'] = gdf['Ra'].clip(0.0, 1.0)
    robot_src = 'Ra'
else:
    gdf['robot_score'] = 0.0
    robot_src = 'fallback_zero'
    print('  WARNING: no robot score column found — robot_score = 0.0')

rs = gdf['robot_score']
print(f'robot_score (source={robot_src}):')
print(f'  mean={rs.mean():.3f}  min={rs.min():.3f}  '
      f'max={rs.max():.3f}  std={rs.std():.3f}')
print(f'  >= 0.45: {(rs >= 0.45).sum()} cells  ({(rs >= 0.45).mean()*100:.1f}%)')
print(f'  >= 0.30: {(rs >= 0.30).sum()} cells  ({(rs >= 0.30).mean()*100:.1f}%)')


robot_score (source=score_robot):
  mean=0.332  min=0.000  max=0.877  std=0.190
  >= 0.45: 306 cells  (15.7%)
  >= 0.30: 1189 cells  (61.1%)


In [5]:
# ══════════════════════════════════════════════════════════
#  service_priority_score — 서비스 우선순위 종합 점수
# ══════════════════════════════════════════════════════════

W_DS    = 0.45
W_DRONE = 0.35
W_ROBOT = 0.20
assert abs(W_DS + W_DRONE + W_ROBOT - 1.0) < 1e-9

# Ds is already in [0,1] from NB08 (min-max normalized demand)
gdf['service_priority_score'] = (
    W_DS    * gdf['Ds'].clip(0.0, 1.0)
  + W_DRONE * gdf['drone_score']
  + W_ROBOT * gdf['robot_score']
).clip(0.0, 1.0)

sps = gdf['service_priority_score']
print(f'service_priority_score:')
print(f'  mean={sps.mean():.3f}  min={sps.min():.3f}  '
      f'max={sps.max():.3f}  std={sps.std():.3f}')
print(f'\n  가중치:')
print(f'    Ds (수요)       × {W_DS}')
print(f'    drone_score     × {W_DRONE}')
print(f'    robot_score     × {W_ROBOT}')


service_priority_score:
  mean=0.428  min=0.278  max=0.782  std=0.078

  가중치:
    Ds (수요)       × 0.45
    drone_score     × 0.35
    robot_score     × 0.2


In [6]:
# ══════════════════════════════════════════════════════════
#  배송 구역 분류 (Delivery Zone Classification)
# ══════════════════════════════════════════════════════════

# high_demand_flag
ds_70p = gdf['Ds'].quantile(0.70)
grade_high = gdf['demand_grade'].isin(['A', 'B']) if 'demand_grade' in gdf.columns \
             else pd.Series(False, index=gdf.index)
gdf['high_demand_flag'] = grade_high | (gdf['Ds'] >= ds_70p)
print(f'high_demand_flag: {gdf["high_demand_flag"].sum()} cells  '
      f'(Ds 70th pct={ds_70p:.3f})')

# Zone classification — priority order matters
def _classify(drone, robot, excl):
    '''Apply zone rules in priority order. Returns (zone_label, zone_reason).'''
    # 1. Hard exclusion first
    if excl or (drone < 0.50 and robot < 0.30):
        return '부적합', '공식 hard exclusion 또는 운용성 부족'
    # 2. Both high → hybrid
    if drone >= 0.65 and robot >= 0.45:
        return '하이브리드', '드론·로봇 모두 양호'
    # 3. Drone primary
    if drone >= 0.65 and robot < 0.45:
        return '드론우선', '드론 적합, 로봇 접근성 낮음'
    # 4. Robot primary
    if robot >= 0.45 and drone < 0.55:
        return '로봇우선', '로봇 접근성 양호, 드론 제약 큼'
    # 5. Either passable → conditional
    if drone >= 0.50 or robot >= 0.30:
        return '보완검토', '수요 또는 제약 조건 보완 검토 필요'
    # Catch-all (should not reach if rule 1 threshold matches)
    return '부적합', '공식 hard exclusion 또는 운용성 부족'

# Vectorised application
excl_series = gdf['hard_exclusion_flag'] if 'hard_exclusion_flag' in gdf.columns \
              else pd.Series(False, index=gdf.index)
results = [
    _classify(d, r, e)
    for d, r, e in zip(gdf['drone_score'], gdf['robot_score'], excl_series)
]
gdf['delivery_zone'] = [z for z, _ in results]
gdf['zone_reason']   = [r for _, r in results]

# priority_zone
OPERABLE_ZONES = {'하이브리드', '드론우선', '로봇우선'}
gdf['priority_zone'] = gdf['high_demand_flag'] & gdf['delivery_zone'].isin(OPERABLE_ZONES)

print(f'\n[Zone counts]')
vc = gdf['delivery_zone'].value_counts()
total = len(gdf)
for z, cnt in vc.items():
    print(f'  {z:<8}  {cnt:3d}  ({cnt/total*100:.1f}%)')
print(f'\n  priority_zone: {gdf["priority_zone"].sum()} cells')

# Warn if all cells fall into one zone
if len(vc) == 1:
    print(f'\n  WARNING: All cells assigned to single zone "{vc.index[0]}"!')
    print('  Check drone_score and robot_score distributions.')


high_demand_flag: 838 cells  (Ds 70th pct=0.344)

[Zone counts]
  부적합       1688  (86.7%)
  드론우선      216  (11.1%)
  하이브리드      43  (2.2%)

  priority_zone: 222 cells


In [7]:
# ══════════════════════════════════════════════════════════
#  공역 프록시 민감도 분석 (Airspace Proxy Sensitivity)
# ══════════════════════════════════════════════════════════

if HAS_PROXY and 'drone_score_with_airspace_proxy' in gdf.columns:
    proxy_results = [
        _classify(d, r, e)
        for d, r, e in zip(
            gdf['drone_score_with_airspace_proxy'],
            gdf['robot_score'],
            excl_series
        )
    ]
    gdf['zone_with_airspace_proxy'] = [z for z, _ in proxy_results]

    # Cells that change zone under proxy
    gdf['airspace_proxy_caution_flag'] = (
        gdf['zone_with_airspace_proxy'] != gdf['delivery_zone']
    )
    n_changed = gdf['airspace_proxy_caution_flag'].sum()
    print(f'[Airspace Proxy Sensitivity]')
    print(f'  공역 프록시 적용 시 구역 변동 셀: {n_changed} / {len(gdf)}')
    if n_changed > 0:
        changed = gdf[gdf['airspace_proxy_caution_flag']][[
            'ADM_NM', 'delivery_zone', 'zone_with_airspace_proxy',
            'drone_score', 'drone_score_with_airspace_proxy'
        ]]
        print(changed.head(10).to_string(index=False))
    print('\n  WARNING: 공식 공역 데이터 없음 — 프록시는 민감도 분석용입니다.')
else:
    gdf['airspace_proxy_caution_flag'] = False
    print('[Airspace Proxy] score_airspace_proxy 컬럼 없음 — 민감도 분석 건너뜀')
    print('  airspace_proxy_caution_flag = False (전체 셀)')


[Airspace Proxy] score_airspace_proxy 컬럼 없음 — 민감도 분석 건너뜀
  airspace_proxy_caution_flag = False (전체 셀)


In [8]:
# ══════════════════════════════════════════════════════════
#  진단 및 요약 (Diagnostics & Summary)
# ══════════════════════════════════════════════════════════

print('=' * 65)
print('NB09b 배송 구역 분류 — 진단 요약')
print('=' * 65)

# Score ranges
print('\n[점수 범위]')
for col in ['drone_score', 'robot_score', 'service_priority_score']:
    s = gdf[col]
    print(f'  {col:<26} [{s.min():.3f}, {s.max():.3f}]  mean={s.mean():.3f}')

# Zone × GU_NM pivot
print('\n[구 × 구역 분포]')
if 'GU_NM' in gdf.columns:
    pivot = (gdf.groupby(['GU_NM', 'delivery_zone'])
               .size()
               .unstack(fill_value=0)
               .assign(합계=lambda df: df.sum(axis=1)))
    print(pivot.to_string())

# Zone summary with mean scores
print('\n[구역별 평균 점수]')
zone_summary = (
    gdf.groupby('delivery_zone').agg(
        cell_count       = ('h3_index',          'count'),
        mean_drone       = ('drone_score',        'mean'),
        mean_robot       = ('robot_score',        'mean'),
        mean_svc         = ('service_priority_score', 'mean'),
        mean_Ds          = ('Ds',                 'mean'),
        priority_cells   = ('priority_zone',      'sum'),
    ).round(3)
)
print(zone_summary.to_string())

# Top 10 by service_priority_score
print('\n[서비스 우선순위 상위 10 셀]')
top10 = gdf.nlargest(10, 'service_priority_score')[[
    'ADM_NM', 'GU_NM', 'delivery_zone', 'service_priority_score',
    'drone_score', 'robot_score', 'Ds', 'priority_zone'
]].round(3)
print(top10.to_string(index=False))

# Bottom 10
print('\n[서비스 우선순위 하위 10 셀]')
bot10 = gdf.nsmallest(10, 'service_priority_score')[[
    'ADM_NM', 'GU_NM', 'delivery_zone', 'service_priority_score',
    'drone_score', 'robot_score', 'Ds', 'priority_zone'
]].round(3)
print(bot10.to_string(index=False))

# Airspace warning
if 'airspace_data_status' in gdf.columns:
    ads = gdf['airspace_data_status'].dropna().unique()
    if any('proxy' in str(s) for s in ads):
        print('\n  WARNING: 공식 공역(Airspace) 데이터 없음.')
        print('  score_airspace = 1.0 (전체 셀). 공역 제약 미반영 상태.')
        print('  "zone_with_airspace_proxy" 컬럼을 민감도 분석에 참고하세요.')


NB09b 배송 구역 분류 — 진단 요약

[점수 범위]
  drone_score                [0.388, 0.982]  mean=0.622
  robot_score                [0.000, 0.877]  mean=0.332
  service_priority_score     [0.278, 0.782]  mean=0.428

[구 × 구역 분포]
delivery_zone  드론우선  부적합  하이브리드   합계
GU_NM                               
분당구             216  703     43  962
수정구               0  621      0  621
중원구               0  364      0  364

[구역별 평균 점수]
               cell_count  mean_drone  mean_robot  mean_svc  mean_Ds  priority_cells
delivery_zone                                                                       
드론우선                  216       0.898       0.217     0.510    0.338             189
부적합                  1688       0.582       0.340     0.414    0.317               0
하이브리드                  43       0.838       0.599     0.559    0.323              33

[서비스 우선순위 상위 10 셀]
ADM_NM GU_NM delivery_zone  service_priority_score  drone_score  robot_score   Ds  priority_zone
  수내1동   분당구           부적합                   0.

In [9]:
# ══════════════════════════════════════════════════════════
#  저장
# ══════════════════════════════════════════════════════════

BASE_OUT_COLS = [
    'h3_index', 'lat', 'lon', 'CSV_ADMI_CD', 'ADM_NM', 'GU_NM',
    'Ds', 'demand_grade',
    'score_airspace', 'score_obstacle', 'score_noise_proxy',
    'score_terrain', 'score_robot', 'score_weather',
    'constraint_score', 'constraint_grade',
    'drone_score', 'robot_score', 'service_priority_score',
    'delivery_zone', 'high_demand_flag', 'priority_zone', 'zone_reason',
    'airspace_data_status', 'airspace_proxy_flag', 'airspace_proxy_caution_flag',
    'hard_exclusion_flag',
]
PROXY_COLS = [
    'score_airspace_proxy',
    'drone_score_with_airspace_proxy',
    'zone_with_airspace_proxy',
    'constraint_score_with_airspace_proxy',
]
extra = [c for c in PROXY_COLS if c in gdf.columns]
FINAL_COLS = [c for c in BASE_OUT_COLS + extra if c in gdf.columns]

out_gdf = gpd.GeoDataFrame(gdf[FINAL_COLS + ['geometry']], crs=gdf.crs)

# Cast booleans and objects to string for GPKG compatibility
for col in out_gdf.select_dtypes(include=['bool']).columns:
    out_gdf[col] = out_gdf[col].astype(str)
for col in out_gdf.select_dtypes(include=['object']).columns:
    if col != 'geometry':
        out_gdf[col] = out_gdf[col].astype(str)

# Drop OGR case-conflict aliases
for col in ['gu_nm', 'adm_nm', 'h3_id']:
    if col in out_gdf.columns:
        out_gdf.drop(columns=[col], inplace=True)

# GPKG
gpkg_path = PROC / 'delivery_zones.gpkg'
out_gdf.to_file(gpkg_path, driver='GPKG')

# CSV
csv_path = PROC / 'delivery_zones.csv'
pd.DataFrame(out_gdf.drop(columns=['geometry'])).to_csv(
    csv_path, index=False, encoding='utf-8-sig'
)

# Zone summary CSV
zone_sum_path = PROC / 'delivery_zone_summary.csv'
zone_summary.reset_index().to_csv(zone_sum_path, index=False, encoding='utf-8-sig')

for p in [gpkg_path, csv_path, zone_sum_path]:
    print(f'  Saved: {p.name}  ({p.stat().st_size // 1024} KB)')


  Saved: delivery_zones.gpkg  (1068 KB)
  Saved: delivery_zones.csv  (672 KB)
  Saved: delivery_zone_summary.csv  (0 KB)


In [10]:
# ══════════════════════════════════════════════════════════
#  지도 시각화 (Folium Map)
# ══════════════════════════════════════════════════════════
try:
    import folium

    ZONE_COLORS = {
        '하이브리드': '#27AE60',   # green
        '드론우선':   '#2980B9',   # blue
        '로봇우선':   '#E67E22',   # orange
        '보완검토':   '#F39C12',   # yellow-amber
        '부적합':     '#95A5A6',   # gray
    }

    cen_lat = gdf['lat'].mean() if 'lat' in gdf.columns else 37.42
    fmap = folium.Map(location=[cen_lat, 127.13], zoom_start=12,
                      tiles='CartoDB positron')

    # Legend
    legend_html = '''
    <div style="position:fixed;bottom:30px;left:30px;z-index:1000;
                background:white;padding:10px;border-radius:8px;
                border:1px solid #ccc;font-size:13px;line-height:1.8">
    <b>배송 구역</b><br>
    <span style="color:#27AE60">&#9632;</span> 하이브리드<br>
    <span style="color:#2980B9">&#9632;</span> 드론우선<br>
    <span style="color:#E67E22">&#9632;</span> 로봇우선<br>
    <span style="color:#F39C12">&#9632;</span> 보완검토<br>
    <span style="color:#95A5A6">&#9632;</span> 부적합
    </div>
    '''
    fmap.get_root().html.add_child(folium.Element(legend_html))

    wgs84 = out_gdf.to_crs('EPSG:4326')
    for _, row in wgs84.iterrows():
        zone   = row.get('delivery_zone', '부적합')
        color  = ZONE_COLORS.get(zone, '#95A5A6')
        adm    = row.get('ADM_NM', '')
        gu     = row.get('GU_NM', '')
        ds_v   = float(row['Ds']) if pd.notna(row.get('Ds')) else float('nan')
        dr_v   = float(row['drone_score']) if pd.notna(row.get('drone_score')) else float('nan')
        rb_v   = float(row['robot_score']) if pd.notna(row.get('robot_score')) else float('nan')
        sp_v   = float(row['service_priority_score']) if pd.notna(row.get('service_priority_score')) else float('nan')
        cs_v   = float(row['constraint_score']) if pd.notna(row.get('constraint_score')) else float('nan')
        reason = row.get('zone_reason', '')

        tip = (f'<b>{adm}</b> ({gu})<br>'
               f'구역: <b>{zone}</b><br>'
               f'서비스점수: {sp_v:.3f}<br>'
               f'수요(Ds): {ds_v:.3f}<br>'
               f'드론점수: {dr_v:.3f}<br>'
               f'로봇점수: {rb_v:.3f}<br>'
               f'제약점수: {cs_v:.3f}<br>'
               f'사유: {reason}')

        folium.GeoJson(
            mapping(row.geometry),
            style_function=lambda feat, c=color: {
                'fillColor': c,
                'color': 'white',
                'weight': 0.4,
                'fillOpacity': 0.75,
            },
            tooltip=folium.Tooltip(tip, sticky=False),
        ).add_to(fmap)

    map_path = PROC / 'delivery_zone_map.html'
    fmap.save(str(map_path))
    print(f'  Map saved: {map_path.name}  ({map_path.stat().st_size // 1024} KB)')
except Exception as e:
    print(f'  Map skipped: {e}')


  Map saved: delivery_zone_map.html  (3607 KB)


In [11]:
# ══════════════════════════════════════════════════════════
#  차트 시각화
# ══════════════════════════════════════════════════════════
try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    import matplotlib.font_manager as fm
    import matplotlib.patches as mpatches

    plt.rcParams['axes.unicode_minus'] = False
    for fname in ['Malgun Gothic', 'NanumGothic']:
        try:
            fm.findfont(fname, fallback_to_default=False)
            plt.rcParams['font.family'] = fname
            break
        except Exception:
            pass

    ZONE_ORDER  = ['하이브리드', '드론우선', '로봇우선', '보완검토', '부적합']
    ZONE_COLORS = {
        '하이브리드': '#27AE60',
        '드론우선':   '#2980B9',
        '로봇우선':   '#E67E22',
        '보완검토':   '#F39C12',
        '부적합':     '#95A5A6',
    }

    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    fig.suptitle('NB09b — 배송 운영 구역 분류 분석', fontsize=13, fontweight='bold')

    # ── Chart 1: Zone bar chart ──────────────────────────
    vc = gdf['delivery_zone'].value_counts()
    zones_ordered = [z for z in ZONE_ORDER if z in vc.index]
    counts = [vc.get(z, 0) for z in zones_ordered]
    colors = [ZONE_COLORS.get(z, '#aaa') for z in zones_ordered]

    bars = axes[0].bar(zones_ordered, counts, color=colors, alpha=0.85, edgecolor='white')
    for bar, cnt in zip(bars, counts):
        axes[0].text(bar.get_x() + bar.get_width() / 2,
                     bar.get_height() + 0.5, str(cnt),
                     ha='center', va='bottom', fontsize=10)
    axes[0].set_title('배송 구역별 H3 셀 수')
    axes[0].set_xlabel('배송 구역')
    axes[0].set_ylabel('H3 셀 수')
    axes[0].grid(axis='y', alpha=0.3)

    # ── Chart 2: drone_score vs robot_score scatter ──────
    scatter_colors = [ZONE_COLORS.get(z, '#aaa') for z in gdf['delivery_zone']]
    axes[1].scatter(
        gdf['drone_score'], gdf['robot_score'],
        c=scatter_colors, alpha=0.65, s=30, edgecolors='none'
    )
    # Threshold reference lines
    axes[1].axvline(0.65, color='gray', ls='--', lw=0.8, alpha=0.5, label='drone 0.65')
    axes[1].axvline(0.50, color='gray', ls=':',  lw=0.8, alpha=0.5, label='drone 0.50')
    axes[1].axhline(0.45, color='gray', ls='--', lw=0.8, alpha=0.5, label='robot 0.45')
    axes[1].axhline(0.30, color='gray', ls=':',  lw=0.8, alpha=0.5, label='robot 0.30')

    patches = [mpatches.Patch(color=ZONE_COLORS[z], label=z)
               for z in ZONE_ORDER if z in gdf['delivery_zone'].values]
    axes[1].legend(handles=patches, fontsize=8, loc='upper left')
    axes[1].set_title('드론 점수 vs 로봇 점수 (구역별)')
    axes[1].set_xlabel('drone_score')
    axes[1].set_ylabel('robot_score')
    axes[1].set_xlim(-0.02, 1.02)
    axes[1].set_ylim(-0.02, 1.02)
    axes[1].grid(alpha=0.25)

    plt.tight_layout()
    chart_path = PROC / 'delivery_zone_chart.png'
    plt.savefig(chart_path, dpi=150, bbox_inches='tight')
    print(f'  Chart saved: {chart_path.name}')
except Exception as e:
    print(f'  Chart skipped: {e}')


  Chart saved: delivery_zone_chart.png


In [12]:
# ══════════════════════════════════════════════════════════
#  최종 검증
# ══════════════════════════════════════════════════════════

print('=' * 65)
print('NB09b 최종 검증')
print('=' * 65)

# Row count vs constraint_grid
cg = gpd.read_file(PROC / 'constraint_grid.gpkg')
print(f'\n  constraint_grid : {len(cg)} rows')
print(f'  delivery_zones  : {len(out_gdf)} rows')
print(f'  Row match: {len(cg) == len(out_gdf)}')

# Null checks
KEY = ['h3_index', 'drone_score', 'robot_score', 'service_priority_score',
       'delivery_zone', 'ADM_NM', 'GU_NM', 'Ds']
print('\n  [Null counts]')
for c in KEY:
    if c in out_gdf.columns:
        n = (out_gdf[c].astype(str) == 'nan').sum()
        flag = '  <-- PROBLEM' if n > 0 else ''
        print(f'    {c:<28} {n}{flag}')

# Score bounds
print('\n  [Score bounds — must be [0,1]]')
all_ok = True
for sc in ['drone_score', 'robot_score', 'service_priority_score']:
    s = gdf[sc]
    ok = (s.min() >= -1e-9) and (s.max() <= 1 + 1e-9)
    if not ok: all_ok = False
    print(f'    {sc:<28} [{s.min():.4f}, {s.max():.4f}]{"  OK" if ok else "  PROBLEM"}')
print(f'  All scores in [0,1]: {all_ok}')

# Zone count warning
print('\n  [Zone distribution]')
vc2 = out_gdf['delivery_zone'].value_counts()
for z, cnt in vc2.items():
    print(f'    {z:<10} {cnt:3d}')
if len(vc2) == 1:
    print('  WARNING: all cells in one zone!')

print('\n✅ NB09b 완료')
print(f'  delivery_zones.gpkg / .csv / _summary.csv')
print(f'  delivery_zone_map.html  /  delivery_zone_chart.png')


NB09b 최종 검증

  constraint_grid : 1947 rows
  delivery_zones  : 1947 rows
  Row match: True

  [Null counts]
    h3_index                     0
    drone_score                  0
    robot_score                  0
    service_priority_score       0
    delivery_zone                0
    ADM_NM                       0
    GU_NM                        0
    Ds                           0

  [Score bounds — must be [0,1]]
    drone_score                  [0.3880, 0.9823]  OK
    robot_score                  [0.0000, 0.8773]  OK
    service_priority_score       [0.2777, 0.7823]  OK
  All scores in [0,1]: True

  [Zone distribution]
    부적합        1688
    드론우선       216
    하이브리드       43

✅ NB09b 완료
  delivery_zones.gpkg / .csv / _summary.csv
  delivery_zone_map.html  /  delivery_zone_chart.png
